# DINO AQUA20 Training Launcher

Launches `main_dino_aqua.py` via `subprocess.Popen`, streams output live to this notebook,
and saves a full log to `{output_dir}/train.log`.

**Note**: Interrupting the kernel does **not** kill the training process — it runs independently.
To kill it after interrupting: `import os, signal; os.kill(<PID>, signal.SIGTERM)`

In [3]:
import subprocess
import os
import sys
import signal
import datetime
from pathlib import Path

In [ ]:
# ── Run configuration ───────────────────────────────────────────────────────
REPO_DIR = Path("/home/alex/internship/dino").resolve()
DATA_PATH = (
    "/home/alex/internship/GradientDistillation/logged_files/distillation/"
    "aqua20/dinov2_vitb/dinov2_vitb_distill_196_ipc1_augs3/data.pth"
)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name  = f"dino-aqua20-{timestamp}"
output_dir = REPO_DIR / "outputs" / run_name
output_dir.mkdir(parents=True, exist_ok=True)

HP = dict(
    arch                        = "vit_small",
    patch_size                  = 16,
    out_dim                     = 65536,
    norm_last_layer             = "true",
    momentum_teacher            = 0.996,
    warmup_teacher_temp         = 0.04,
    teacher_temp                = 0.07,
    warmup_teacher_temp_epochs  = 30,
    use_fp16                    = "true",
    weight_decay                = 0.04,
    weight_decay_end            = 0.4,
    clip_grad                   = 3.0,
    batch_size_per_gpu          = 64,
    epochs                      = 100,
    warmup_epochs               = 10,
    lr                          = 0.0005,
    min_lr                      = 1e-6,
    optimizer                   = "adamw",
    drop_path_rate              = 0.1,
    freeze_last_layer           = 1,
    global_crops_scale          = "0.4 1.0",
    local_crops_number          = 8,
    local_crops_scale           = "0.05 0.4",
    num_workers                 = 4,
    saveckp_freq                = 20,
    seed                        = 0,
    # kNN eval
    knn_eval_freq               = 10,
    knn_nb_knn                  = "1 5 20",
    knn_temperature             = 0.07,
    knn_test_data_path          = '/home/alex/internship/datasets/aqua20/data/aqua20/test',
    num_classes                 = 20,
    # W&B
    use_wandb                   = "true",
    wandb_project               = "dino-aqua20",
    wandb_run_name              = run_name,
)

print(f"Run name : {run_name}")
print(f"Output   : {output_dir}")
print(f"W&B name : {HP['wandb_run_name']}")

Run name : dino-aqua20-20260505_121606
Output   : /home/alex/internship/dino/outputs/dino-aqua20-20260505_121606
W&B name : dino-aqua20-20260505_121606


In [ ]:
# ── Build command ────────────────────────────────────────────────────────────
# Multi-value args (lists) are passed as space-separated strings in HP;
# we split them when building the cmd list.
MULTI_VALUE_ARGS = {"global_crops_scale", "local_crops_scale", "knn_nb_knn"}

cmd = [
    "uv", "run",
    "-m", "torch.distributed.launch",
    "--nproc_per_node=1",
    "main_dino_aqua.py",
    "--distilled_data_path", DATA_PATH,
    "--output_dir", str(output_dir),
]
# Add dist_url with unique file to avoid collisions with other runs
cmd += ["--dist_url", f"file:///tmp/dino_dist_{timestamp}"]

for key, val in HP.items():
    if key in MULTI_VALUE_ARGS:
        cmd += [f"--{key}"] + str(val).split()
    else:
        cmd += [f"--{key}", str(val)]

print("Command:")
print(" \\".join(["  " + c for c in cmd]))

Started PID=15786  log=/home/alex/internship/dino/outputs/dino-aqua20-20260505_120658/train.log
wandb: Syncing run dino-aqua20-20260505_120658
wandb: View run at https://wandb.ai/alex26delaveau-lyon-2-/dino-aqua20/runs/0zvrv54f
Data loaded: there are 200 images.
Student and Teacher are built: they are both vit_small network.
Starting DINO training !
Epoch: [0/100] Total time: 0:00:59  loss: 10.796501
Epoch: [1/100] Total time: 0:00:21  loss: 10.854280
Epoch: [2/100] Total time: 0:00:22  loss: 10.958606
(training confirmed running, streaming loop stopped for notebook save)


In [ ]:
# ── Launch training and stream output ────────────────────────────────────────
# stdout + stderr are merged and streamed line-by-line:
#   - printed to this cell (visible in notebook)
#   - written to {output_dir}/train.log
#
# Interrupting the kernel stops the streaming loop but NOT the subprocess.
# The process keeps running and logs keep accumulating in train.log.

log_path = output_dir / "train.log"

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = "/usr/lib/wsl/lib:" + env.get("LD_LIBRARY_PATH", "")

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=str(REPO_DIR),
    env=env,
)

print(f"Started PID={process.pid}  log={log_path}")
print("=" * 70)

with open(log_path, "w") as log_file:
    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

process.wait()
print(f"\nProcess exited with code {process.returncode}")

Started PID=21157  log=/home/alex/internship/dino/outputs/dino-aqua20-20260505_121606/train.log
wandb: Syncing run dino-aqua20-20260505_121606
Data loaded: there are 200 images.
Starting DINO training !
Epoch: [0/100]  loss: 10.796499
...
Epoch 9 | 10-NN (k=10, bank=20, queries=1612): top1=15.3%  top5=45.4%  F1_macro=6.2%  F1_weighted=16.7%


In [ ]:
# ── Run info & kill instructions ─────────────────────────────────────────────
print(f"PID         : {process.pid}")
print(f"Output dir  : {output_dir}")
print(f"Log file    : {log_path}")
print(f"W&B project : {HP['wandb_project']}")
print(f"W&B run     : {HP['wandb_run_name']}")
print()
print("To tail the log from a terminal:")
print(f"  tail -f {log_path}")
print()
print("To kill the run if kernel was interrupted:")
print(f"  import os, signal; os.kill({process.pid}, signal.SIGTERM)")

PID         : 21157
Output dir  : /home/alex/internship/dino/outputs/dino-aqua20-20260505_121606
Log file    : /home/alex/internship/dino/outputs/dino-aqua20-20260505_121606/train.log
W&B project : dino-aqua20
W&B run     : dino-aqua20-20260505_121606

To tail the log from a terminal:
  tail -f /home/alex/internship/dino/outputs/dino-aqua20-20260505_121606/train.log

To kill the run if kernel was interrupted:
  import os, signal; os.kill(21157, signal.SIGTERM)


---

## Real Dataset Run

Trains on the full AQUA20 train set (6559 images) instead of the distilled set.
kNN memory bank = training set; queries = test set.

In [4]:
# ── Real-dataset run configuration ────────────────────────────────────────────
REPO_DIR_R = Path("/home/alex/internship/dino").resolve()
DATA_PATH_REAL = "/home/alex/internship/datasets/aqua20/data/aqua20/train"

timestamp_r = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name_r  = f"dino-aqua20-real-{timestamp_r}"
output_dir_r = REPO_DIR_R / "outputs" / run_name_r
output_dir_r.mkdir(parents=True, exist_ok=True)

HP_R = dict(
    arch                        = "vit_small",
    patch_size                  = 16,
    out_dim                     = 65536,
    norm_last_layer             = "true",
    momentum_teacher            = 0.996,
    warmup_teacher_temp         = 0.04,
    teacher_temp                = 0.07,
    warmup_teacher_temp_epochs  = 30,
    use_fp16                    = "true",
    weight_decay                = 0.04,
    weight_decay_end            = 0.4,
    clip_grad                   = 3.0,
    batch_size_per_gpu          = 64,
    epochs                      = 100,
    warmup_epochs               = 10,
    lr                          = 0.0005,
    min_lr                      = 1e-6,
    optimizer                   = "adamw",
    drop_path_rate              = 0.1,
    freeze_last_layer           = 1,
    global_crops_scale          = "0.4 1.0",
    local_crops_number          = 8,
    local_crops_scale           = "0.05 0.4",
    num_workers                 = 4,
    saveckp_freq                = 20,
    seed                        = 0,
    # kNN eval (memory bank = full train set, so k=1/5/20 are all distinct)
    knn_eval_freq               = 10,
    knn_nb_knn                  = "1 5 20",
    knn_temperature             = 0.07,
    knn_test_data_path          = '/home/alex/internship/datasets/aqua20/data/aqua20/test',
    num_classes                 = 20,
    # W&B
    use_wandb                   = "true",
    wandb_project               = "dino-aqua20",
    wandb_run_name              = run_name_r,
)

print(f"Run name : {run_name_r}")
print(f"Output   : {output_dir_r}")
print(f"W&B name : {HP_R['wandb_run_name']}")

Run name : dino-aqua20-real-20260505_131915
Output   : /home/alex/internship/dino/outputs/dino-aqua20-real-20260505_131915
W&B name : dino-aqua20-real-20260505_131915


In [5]:
# ── Build command (real dataset) ──────────────────────────────────────────────
MULTI_VALUE_ARGS_R = {"global_crops_scale", "local_crops_scale", "knn_nb_knn"}

cmd_r = [
    "uv", "run",
    "-m", "torch.distributed.launch",
    "--nproc_per_node=1",
    "main_dino_aqua.py",
    "--data_path", DATA_PATH_REAL,
    "--output_dir", str(output_dir_r),
]
cmd_r += ["--dist_url", f"file:///tmp/dino_dist_{timestamp_r}"]

for key, val in HP_R.items():
    if key in MULTI_VALUE_ARGS_R:
        cmd_r += [f"--{key}"] + str(val).split()
    else:
        cmd_r += [f"--{key}", str(val)]

print("Command:")
print(" \\".join(["  " + c for c in cmd_r]))

Command:
  uv \  run \  -m \  torch.distributed.launch \  --nproc_per_node=1 \  main_dino_aqua.py \  --data_path \  /home/alex/internship/datasets/aqua20/data/aqua20/train \  --output_dir \  /home/alex/internship/dino/outputs/dino-aqua20-real-20260505_131915 \  --dist_url \  file:///tmp/dino_dist_20260505_131915 \  --arch \  vit_small \  --patch_size \  16 \  --out_dim \  65536 \  --norm_last_layer \  true \  --momentum_teacher \  0.996 \  --warmup_teacher_temp \  0.04 \  --teacher_temp \  0.07 \  --warmup_teacher_temp_epochs \  30 \  --use_fp16 \  true \  --weight_decay \  0.04 \  --weight_decay_end \  0.4 \  --clip_grad \  3.0 \  --batch_size_per_gpu \  64 \  --epochs \  100 \  --warmup_epochs \  10 \  --lr \  0.0005 \  --min_lr \  1e-06 \  --optimizer \  adamw \  --drop_path_rate \  0.1 \  --freeze_last_layer \  1 \  --global_crops_scale \  0.4 \  1.0 \  --local_crops_number \  8 \  --local_crops_scale \  0.05 \  0.4 \  --num_workers \  4 \  --saveckp_freq \  20 \  --seed \  0 \  --

In [6]:
# ── Launch training and stream output (real dataset) ───────────────────────
log_path_r = output_dir_r / "train.log"

env_r = os.environ.copy()
env_r["LD_LIBRARY_PATH"] = "/usr/lib/wsl/lib:" + env_r.get("LD_LIBRARY_PATH", "")

process_r = subprocess.Popen(
    cmd_r,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=str(REPO_DIR_R),
    env=env_r,
)

print(f"Started PID={process_r.pid}  log={log_path_r}")
print("=" * 70)

with open(log_path_r, "w") as log_file:
    for line in process_r.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

process_r.wait()
print(f"\nProcess exited with code {process_r.returncode}")

Started PID=51909  log=/home/alex/internship/dino/outputs/dino-aqua20-real-20260505_131915/train.log
/home/alex/internship/dino/.venv/lib/python3.8/site-packages/torch/distributed/launch.py:180: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use_env is set by default in torchrun.
If your script expects `--local_rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  warnings.warn(
Using cache found in /home/alex/.cache/torch/hub/facebookresearch_xcit_main
| distributed init (rank 0): file:///tmp/dino_dist_20260505_131915
git:
  sha: 7f655fde3289697ef40f1874d5cdb66484abf4c8, status: has uncommited changes, branch: feat/wandb-knn-eval

arch: vit_small
batch_size_per_gpu: 64
clip_grad: 3.0
data_path: /home/alex/internship/datasets/aqua20/data/aqua20/train
dist_url: file:///tmp/dino_d

KeyboardInterrupt: 

In [ ]:
# ── Run info & kill instructions (real dataset) ───────────────────────────
print(f"PID         : {process_r.pid}")
print(f"Output dir  : {output_dir_r}")
print(f"Log file    : {log_path_r}")
print(f"W&B project : {HP_R['wandb_project']}")
print(f"W&B run     : {HP_R['wandb_run_name']}")
print()
print("To tail the log from a terminal:")
print(f"  tail -f {log_path_r}")
print()
print("To kill the run if kernel was interrupted:")
print(f"  import os, signal; os.kill({process_r.pid}, signal.SIGTERM)")

### Distilled No Repeat

In [10]:
# ── Run configuration ───────────────────────────────────────────────────────
REPO_DIR = Path("/home/alex/internship/dino").resolve()
DATA_PATH = (
    "/home/alex/internship/GradientDistillation/logged_files/distillation/"
    "aqua20/dinov2_vitb/dinov2_vitb_distill_196_ipc1_augs3/data.pth"
)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name  = f"dino-no-repeat-aqua20-{timestamp}"
output_dir = REPO_DIR / "outputs" / run_name
output_dir.mkdir(parents=True, exist_ok=True)

HP = dict(
    arch                        = "vit_small",
    patch_size                  = 16,
    out_dim                     = 65536,
    norm_last_layer             = "true",
    momentum_teacher            = 0.996,
    warmup_teacher_temp         = 0.04,
    teacher_temp                = 0.07,
    warmup_teacher_temp_epochs  = 30,
    use_fp16                    = "true",
    weight_decay                = 0.04,
    weight_decay_end            = 0.4,
    clip_grad                   = 3.0,
    batch_size_per_gpu          = 20,
    epochs                      = 100,
    warmup_epochs               = 10,
    lr                          = 0.0005,
    min_lr                      = 1e-6,
    optimizer                   = "adamw",
    drop_path_rate              = 0.1,
    freeze_last_layer           = 1,
    global_crops_scale          = "0.4 1.0",
    local_crops_number          = 8,
    local_crops_scale           = "0.05 0.4",
    num_workers                 = 4,
    saveckp_freq                = 20,
    seed                        = 0,
    # kNN eval
    knn_eval_freq               = 10,
    knn_nb_knn                  = "1 5 20",
    knn_temperature             = 0.07,
    knn_test_data_path          = '/home/alex/internship/datasets/aqua20/data/aqua20/test',
    num_classes                 = 20,
    # W&B
    use_wandb                   = "true",
    wandb_project               = "dino-aqua20",
    wandb_run_name              = run_name,

    #Distilled dataset
    num_dataset_repeats         = 1,
)

print(f"Run name : {run_name}")
print(f"Output   : {output_dir}")
print(f"W&B name : {HP['wandb_run_name']}")

Run name : dino-no-repeat-aqua20-20260505_180527
Output   : /home/alex/internship/dino/outputs/dino-no-repeat-aqua20-20260505_180527
W&B name : dino-no-repeat-aqua20-20260505_180527


In [11]:
# ── Build command ────────────────────────────────────────────────────────────
# Multi-value args (lists) are passed as space-separated strings in HP;
# we split them when building the cmd list.
MULTI_VALUE_ARGS = {"global_crops_scale", "local_crops_scale", "knn_nb_knn"}

cmd = [
    "uv", "run",
    "-m", "torch.distributed.launch",
    "--nproc_per_node=1",
    "main_dino_aqua.py",
    "--distilled_data_path", DATA_PATH,
    "--output_dir", str(output_dir),
]
# Add dist_url with unique file to avoid collisions with other runs
cmd += ["--dist_url", f"file:///tmp/dino_dist_{timestamp}"]

for key, val in HP.items():
    if key in MULTI_VALUE_ARGS:
        cmd += [f"--{key}"] + str(val).split()
    else:
        cmd += [f"--{key}", str(val)]

print("Command:")
print(" \\".join(["  " + c for c in cmd]))

Command:
  uv \  run \  -m \  torch.distributed.launch \  --nproc_per_node=1 \  main_dino_aqua.py \  --distilled_data_path \  /home/alex/internship/GradientDistillation/logged_files/distillation/aqua20/dinov2_vitb/dinov2_vitb_distill_196_ipc1_augs3/data.pth \  --output_dir \  /home/alex/internship/dino/outputs/dino-no-repeat-aqua20-20260505_180527 \  --dist_url \  file:///tmp/dino_dist_20260505_180527 \  --arch \  vit_small \  --patch_size \  16 \  --out_dim \  65536 \  --norm_last_layer \  true \  --momentum_teacher \  0.996 \  --warmup_teacher_temp \  0.04 \  --teacher_temp \  0.07 \  --warmup_teacher_temp_epochs \  30 \  --use_fp16 \  true \  --weight_decay \  0.04 \  --weight_decay_end \  0.4 \  --clip_grad \  3.0 \  --batch_size_per_gpu \  20 \  --epochs \  100 \  --warmup_epochs \  10 \  --lr \  0.0005 \  --min_lr \  1e-06 \  --optimizer \  adamw \  --drop_path_rate \  0.1 \  --freeze_last_layer \  1 \  --global_crops_scale \  0.4 \  1.0 \  --local_crops_number \  8 \  --local_cr

In [12]:
# ── Launch training and stream output ────────────────────────────────────────
# stdout + stderr are merged and streamed line-by-line:
#   - printed to this cell (visible in notebook)
#   - written to {output_dir}/train.log
#
# Interrupting the kernel stops the streaming loop but NOT the subprocess.
# The process keeps running and logs keep accumulating in train.log.

log_path = output_dir / "train.log"

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = "/usr/lib/wsl/lib:" + env.get("LD_LIBRARY_PATH", "")

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=str(REPO_DIR),
    env=env,
)

print(f"Started PID={process.pid}  log={log_path}")
print("=" * 70)

with open(log_path, "w") as log_file:
    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

process.wait()
print(f"\nProcess exited with code {process.returncode}")

Started PID=169863  log=/home/alex/internship/dino/outputs/dino-no-repeat-aqua20-20260505_180527/train.log
/home/alex/internship/dino/.venv/lib/python3.8/site-packages/torch/distributed/launch.py:180: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use_env is set by default in torchrun.
If your script expects `--local_rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  warnings.warn(
Using cache found in /home/alex/.cache/torch/hub/facebookresearch_xcit_main
| distributed init (rank 0): file:///tmp/dino_dist_20260505_180527
git:
  sha: 7f655fde3289697ef40f1874d5cdb66484abf4c8, status: has uncommited changes, branch: feat/wandb-knn-eval

arch: vit_small
batch_size_per_gpu: 20
clip_grad: 3.0
data_path: /path/to/imagenet/train/
dist_url: file:///tmp/dino_dist_20260505_180527
disti

In [13]:
# ── Run info & kill instructions ─────────────────────────────────────────────
print(f"PID         : {process.pid}")
print(f"Output dir  : {output_dir}")
print(f"Log file    : {log_path}")
print(f"W&B project : {HP['wandb_project']}")
print(f"W&B run     : {HP['wandb_run_name']}")
print()
print("To tail the log from a terminal:")
print(f"  tail -f {log_path}")
print()
print("To kill the run if kernel was interrupted:")
print(f"  import os, signal; os.kill({process.pid}, signal.SIGTERM)")

PID         : 169863
Output dir  : /home/alex/internship/dino/outputs/dino-no-repeat-aqua20-20260505_180527
Log file    : /home/alex/internship/dino/outputs/dino-no-repeat-aqua20-20260505_180527/train.log
W&B project : dino-aqua20
W&B run     : dino-no-repeat-aqua20-20260505_180527

To tail the log from a terminal:
  tail -f /home/alex/internship/dino/outputs/dino-no-repeat-aqua20-20260505_180527/train.log

To kill the run if kernel was interrupted:
  import os, signal; os.kill(169863, signal.SIGTERM)
